## split 1--deepseek_v3  ...pre_tokenizer

In [1]:
# code the split 1.
import regex

corpus = """
I bought 5 apples and 12 oranges today
东京塔 is 333 meters tall
彼女は3冊の本を持っている
The meeting is scheduled for 7pm in 会議室
价格是99元 for the ticket
She has 2 cats and 1 dog
この漫画は42巻まで出版されている
北京大学 was founded in year 189
Room 101 is next to 図書館
总共有365天 in a year
"""

In [2]:
documents = [line for line in corpus.split("\n") if line.strip()]
documents

['I bought 5 apples and 12 oranges today',
 '东京塔 is 333 meters tall',
 '彼女は3冊の本を持っている',
 'The meeting is scheduled for 7pm in 会議室',
 '价格是99元 for the ticket',
 'She has 2 cats and 1 dog',
 'この漫画は42巻まで出版されている',
 '北京大学 was founded in year 189',
 'Room 101 is next to 図書館',
 '总共有365天 in a year']

In [3]:
pattern = regex.compile(r"\p{N}{1,3}") # deepseekv3 tokenzier--pre_tokenize split#1
pattern

regex.Regex('\\p{N}{1,3}', flags=regex.V0)

In [4]:

lst = [[index, match.group(), match.span()] for index, document in enumerate(documents) for match in pattern.finditer(document)]
lst


[[0, '5', (9, 10)],
 [0, '12', (22, 24)],
 [1, '333', (7, 10)],
 [2, '3', (3, 4)],
 [3, '7', (29, 30)],
 [4, '99', (3, 5)],
 [5, '2', (8, 9)],
 [5, '1', (19, 20)],
 [6, '42', (5, 7)],
 [7, '189', (25, 28)],
 [8, '101', (5, 8)],
 [9, '365', (3, 6)]]

In [5]:
indices_to_change = [index[0] for index in lst]
start_end = [index[2] for index in lst]


In [6]:
indices_to_change

[0, 0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 9]

In [7]:
start_end

[(9, 10),
 (22, 24),
 (7, 10),
 (3, 4),
 (29, 30),
 (3, 5),
 (8, 9),
 (19, 20),
 (5, 7),
 (25, 28),
 (5, 8),
 (3, 6)]

In [8]:
start, end = start_end[1]
start, end

(22, 24)

In [9]:
len(start_end[0])

2

In [10]:

def split1_transform(documents, indices_to_change, start_end):

    transformed_documents = []
    span_index = 0
    for document_index, document in enumerate(documents):

        temp_doc = []
        previous_end = 0
        while (span_index < len(start_end) and document_index == indices_to_change[span_index]):
            start, end = start_end[span_index]

            if start > previous_end:
                temp_doc.append(document[previous_end: start])
                previous_end = start

            temp_doc.append(document[start:end])
            previous_end = end

            span_index += 1

        if previous_end < len(document):
            temp_doc.append(document[previous_end:])

        transformed_documents.append(temp_doc)


    return transformed_documents

    

In [11]:
split1_transformed_data = split1_transform(
    documents,
    indices_to_change,
    start_end,
)
split1_transformed_data

[['I bought ', '5', ' apples and ', '12', ' oranges today'],
 ['东京塔 is ', '333', ' meters tall'],
 ['彼女は', '3', '冊の本を持っている'],
 ['The meeting is scheduled for ', '7', 'pm in 会議室'],
 ['价格是', '99', '元 for the ticket'],
 ['She has ', '2', ' cats and ', '1', ' dog'],
 ['この漫画は', '42', '巻まで出版されている'],
 ['北京大学 was founded in year ', '189'],
 ['Room ', '101', ' is next to 図書館'],
 ['总共有', '365', '天 in a year']]

In [33]:
split1_transformed_data[9][2]

'天 in a year'

In [12]:
for i in split1_transformed_data:
    print(i)

['I bought ', '5', ' apples and ', '12', ' oranges today']
['东京塔 is ', '333', ' meters tall']
['彼女は', '3', '冊の本を持っている']
['The meeting is scheduled for ', '7', 'pm in 会議室']
['价格是', '99', '元 for the ticket']
['She has ', '2', ' cats and ', '1', ' dog']
['この漫画は', '42', '巻まで出版されている']
['北京大学 was founded in year ', '189']
['Room ', '101', ' is next to 図書館']
['总共有', '365', '天 in a year']


## split 2

In [26]:
for document in split1_transformed_data:
    for piece in document:
        print(piece)
    break

I bought 
5
 apples and 
12
 oranges today


In [13]:
import regex

cjk_pattern = regex.compile("[一-龥\u3040-\u309F\u30A0-\u30FF]+")
cjk_pattern

regex.Regex('[一-龥\u3040-ゟ゠-ヿ]+', flags=regex.V0)

In [ ]:

def split2_transform(split1_documents):

    transformed_documents = []

    for document in split1_documents:
        temp_doc = []

        for piece in document:
            previous_end = 0

            for match in cjk_pattern.finditer(piece):
                start, end = match.span()

                if start > previous_end:
                    temp_doc.append([piece[previous_end:start]])

                temp_doc.append(piece[start:end])
                previous_end = end


            if previous_end < len(piece):
                temp_doc.append(piece[previous_end:])

        transformed_documents.append(temp_doc)

    return transformed_documents
                    

In [37]:
split2_transformed_data = split2_transform(split1_transformed_data)
split2_transformed_data

[['I bought ', '5', ' apples and ', '12', ' oranges today'],
 ['东京塔', ' is ', '333', ' meters tall'],
 ['彼女は', '3', '冊の本を持っている'],
 ['The meeting is scheduled for ', '7', ['pm in '], '会議室'],
 ['价格是', '99', '元', ' for the ticket'],
 ['She has ', '2', ' cats and ', '1', ' dog'],
 ['この漫画は', '42', '巻まで出版されている'],
 ['北京大学', ' was founded in year ', '189'],
 ['Room ', '101', [' is next to '], '図書館'],
 ['总共有', '365', '天', ' in a year']]

## Split 3